# CRANE online demo

This notebook runs a small CRANE demo in the online environment. It uses the GSC CRISPR demo data included in this repository.

Run the cells from top to bottom.

In [ ]:
!python examples/rebuild_demo_data.py --dataset gsc

In [ ]:
import scanpy as sc
import crane

adata = sc.read_h5ad("demo_workspace/data/demo_gsc.h5ad")
adata

## Standard Scanpy view

This first creates a regular exploratory Scanpy view of the same data. CRANE below still uses the original expression layer, not the HVG-subsetted object.

In [ ]:
view = adata.copy()
sc.pp.highly_variable_genes(
    view,
    layer="log1p_norm",
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5,
)
view_hvg = view[:, view.var["highly_variable"]].copy()
view_hvg.X = view_hvg.layers["log1p_norm"].copy()
sc.pp.pca(view_hvg, n_comps=30, random_state=0)
sc.pp.neighbors(view_hvg, n_neighbors=20, n_pcs=30, metric="cosine")
sc.tl.umap(view_hvg, random_state=0)
sc.pl.umap(view_hvg, color=["perturbation_targets", "S_score", "G2M_score", "phase"], frameon=False)

## Gene response

CRANE scores genes by perturbation response in the GSC CRISPR data.

In [ ]:
gene_result = crane.tl.gene_response(
    adata,
    perturbation_key="perturbation_targets",
    control_value="control",
    case_value="GSC",
    layer="log1p_norm",
    inplace=False,
)

gene_result.summary(normalized=True).head(10)

In [ ]:
gene_result.gene_pair().iloc[:5, :5]

In [ ]:
gene_result.gene_module()[["module_label"]].head(10)

## Cell response

CRANE can also score the perturbation response at the cell level.

In [ ]:
cell_result = crane.tl.cell_response(
    adata,
    perturbation_key="perturbation_targets",
    control_value="control",
    case_value="GSC",
    layer="log1p_norm",
    inplace=False,
)

cell_result.summary().head(10)

## Extension response

The extension demo compares cell-level covariate structure on a standard Scanpy graph and on the CRANE response graph.

In [ ]:
!python examples/extension_response_covariates.py

## Functional response

The Erlotinib drug-data functional demo is larger. Run the next cells if you want to test MAPK and EGFR gene-set response online.

In [ ]:
!python examples/rebuild_demo_data.py --dataset erlotinib

In [ ]:
!python examples/function_response_erlotinib.py